In [1]:
import pandas as pd
!pip install pyspark
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("MergeParquet").getOrCreate()

In [2]:
files = [
    "/content/drive/MyDrive/NYC Yellow Taxi 2025 Dataset/yellow_tripdata_2025-01.parquet",
    "/content/drive/MyDrive/NYC Yellow Taxi 2025 Dataset/yellow_tripdata_2025-02.parquet",
    "/content/drive/MyDrive/NYC Yellow Taxi 2025 Dataset/yellow_tripdata_2025-03.parquet",
    "/content/drive/MyDrive/NYC Yellow Taxi 2025 Dataset/yellow_tripdata_2025-04.parquet",
    "/content/drive/MyDrive/NYC Yellow Taxi 2025 Dataset/yellow_tripdata_2025-05.parquet",
    "/content/drive/MyDrive/NYC Yellow Taxi 2025 Dataset/yellow_tripdata_2025-06.parquet",
    "/content/drive/MyDrive/NYC Yellow Taxi 2025 Dataset/yellow_tripdata_2025-07.parquet",
    "/content/drive/MyDrive/NYC Yellow Taxi 2025 Dataset/yellow_tripdata_2025-08.parquet",
    "/content/drive/MyDrive/NYC Yellow Taxi 2025 Dataset/yellow_tripdata_2025-09.parquet",
    "/content/drive/MyDrive/NYC Yellow Taxi 2025 Dataset/yellow_tripdata_2025-10.parquet",
    "/content/drive/MyDrive/NYC Yellow Taxi 2025 Dataset/yellow_tripdata_2025-11.parquet",
    "/content/drive/MyDrive/NYC Yellow Taxi 2025 Dataset/yellow_tripdata_2025-12.parquet"
]
dt = spark.read.parquet(*files)

In [3]:
# dfs = []
# for file in files:
#     dfs.append(pd.read_parquet(file))
# df=pd.concat(dfs,ignore_index=True)


# the session is getting crashed

In [4]:
# first = True
# for file in files:
#     df = pd.read_parquet(file)
#     df.to_csv(
#         "/content/drive/MyDrive/combined.csv",
#         mode="a",
#         header=first,
#         index=False
#     )
#     first = False

In [5]:
dt.coalesce(1).write.mode("overwrite").option("header", True).csv("/content/drive/MyDrive/taxi_combined_csv")

In [6]:
df = spark.read.option("header", True).csv("/content/drive/MyDrive/taxi_combined_csv")

In [7]:
df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       1|2025-05-01T00:07:...| 2025-05-01T00:24:...|              1|          3.7|         1|                 N|         140|    

In [8]:
df.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee',
 'cbd_congestion_fee']

In [9]:
df.printSchema()

root
 |-- VendorID: string (nullable = true)
 |-- tpep_pickup_datetime: string (nullable = true)
 |-- tpep_dropoff_datetime: string (nullable = true)
 |-- passenger_count: string (nullable = true)
 |-- trip_distance: string (nullable = true)
 |-- RatecodeID: string (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: string (nullable = true)
 |-- DOLocationID: string (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- fare_amount: string (nullable = true)
 |-- extra: string (nullable = true)
 |-- mta_tax: string (nullable = true)
 |-- tip_amount: string (nullable = true)
 |-- tolls_amount: string (nullable = true)
 |-- improvement_surcharge: string (nullable = true)
 |-- total_amount: string (nullable = true)
 |-- congestion_surcharge: string (nullable = true)
 |-- Airport_fee: string (nullable = true)
 |-- cbd_congestion_fee: string (nullable = true)



In [10]:
print(df.count())

48722602


In [11]:
from pyspark.sql.functions import col, count, when

In [12]:
df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       0|                   0|                    0|       11611894|            0|  11611894|          11611894|           0|    

In [14]:
df = df.dropDuplicates()

In [16]:
from pyspark.sql.functions import trim

df = df.withColumn(
    "store_and_fwd_flag",
    trim(col("store_and_fwd_flag"))
)

In [17]:
from pyspark.sql.functions import col

df = df.withColumn("passenger_count", col("passenger_count").cast("int"))

df = df.withColumn("trip_distance", col("trip_distance").cast("double"))

df = df.withColumn("fare_amount", col("fare_amount").cast("double"))

df = df.withColumn("tip_amount", col("tip_amount").cast("double"))

df = df.withColumn("total_amount", col("total_amount").cast("double"))

In [18]:
df.printSchema()

root
 |-- VendorID: string (nullable = true)
 |-- tpep_pickup_datetime: string (nullable = true)
 |-- tpep_dropoff_datetime: string (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: string (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: string (nullable = true)
 |-- DOLocationID: string (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: string (nullable = true)
 |-- mta_tax: string (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: string (nullable = true)
 |-- improvement_surcharge: string (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: string (nullable = true)
 |-- Airport_fee: string (nullable = true)
 |-- cbd_congestion_fee: string (nullable = true)



In [19]:
from pyspark.sql.functions import to_timestamp

df = df.withColumn(
    "tpep_pickup_datetime",
    to_timestamp("tpep_pickup_datetime")
)

df = df.withColumn(
    "tpep_dropoff_datetime",
    to_timestamp("tpep_dropoff_datetime")
)

In [20]:
df.printSchema()

root
 |-- VendorID: string (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: string (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: string (nullable = true)
 |-- DOLocationID: string (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: string (nullable = true)
 |-- mta_tax: string (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: string (nullable = true)
 |-- improvement_surcharge: string (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: string (nullable = true)
 |-- Airport_fee: string (nullable = true)
 |-- cbd_congestion_fee: string (nullable = true)



In [21]:
df = df.na.drop(subset=[
    "VendorID",
    "PULocationID",
    "DOLocationID",
    "fare_amount"
])

In [22]:
df = df.na.fill({
    "passenger_count": 1
})

In [23]:
df = df.filter(col("fare_amount") >= 0)

df = df.filter(col("trip_distance") >= 0)

df = df.filter(col("passenger_count") >= 0)

In [24]:
df = df.filter(
    col("tpep_dropoff_datetime") >
    col("tpep_pickup_datetime")
)

In [25]:
df.select("VendorID").distinct().show()

+--------+
|VendorID|
+--------+
|       1|
|       2|
|       6|
+--------+



In [26]:
df.select("payment_type").distinct().show()

+------------+
|payment_type|
+------------+
|           3|
|           1|
|           4|
|           2|
|           0|
|           5|
+------------+



In [27]:
df.select("RatecodeID").distinct().show()

+----------+
|RatecodeID|
+----------+
|         3|
|        99|
|         5|
|         6|
|         1|
|         4|
|         2|
|      NULL|
+----------+



In [32]:
from pyspark.sql.functions import hour, month, dayofweek

df = df.withColumn(
    "pickup_hour",
    hour("tpep_pickup_datetime")
)

df = df.withColumn(
    "pickup_month",
    month("tpep_pickup_datetime")
)

df = df.withColumn(
    "day_of_week",
    dayofweek("tpep_pickup_datetime")
)

In [33]:
from pyspark.sql.functions import unix_timestamp

df = df.withColumn(
    "trip_minutes",
    (
        unix_timestamp("tpep_dropoff_datetime") -
        unix_timestamp("tpep_pickup_datetime")
    ) / 60
)

In [ ]:
df.write.mode("overwrite").parquet(
    "/content/drive/MyDrive/clean_taxi_data"
)